# Step 4 · PINO — the FNO plus physics (bonus)

The PINO is the FNO architecture trained with the PINN's physics loss added
($L = L_\text{data} + \lambda\,L_\text{pde} + \lambda\,L_\text{ic}$). It reaches a much lower
PDE residual than the FNO, and it supports test-time **"Refine with physics"**:
starting from the PINO weights, minimize the residual for one new IC (no data) —
the bridge between the slow PINN and the fast FNO. Thin wrapper over `shocklab`;
to (re)train: `python -m shocklab.train.pino --config configs/pino.yaml`.

In [ ]:
import torch

from shocklab.train.common import get_device
from shocklab.train.pino import PINOConfig, build_model, evaluate, residual_floor

cfg = PINOConfig.from_yaml("configs/pino.yaml")
device = get_device("cuda")
model = build_model(cfg).to(device).eval()
model.load_state_dict(torch.load("checkpoints/pino/model.pt", map_location=device))
R0 = residual_floor(cfg.floor_samples, cfg.floor_seed, cfg.nt_fine)
print(f"truth residual floor R0 = {R0:.3e}")

In [ ]:
# residual vs the FNO, data error, and Refine-with-physics on the OOD presets
res = evaluate(model, device, cfg, R0)
print(f"PINO residual / FNO residual = {res['residual_ratio_pino_over_fno']:.2f}  (lower is better)")
print(f"PINO residual / floor        = {res['residual_pino_over_floor']:.2f}")
print(f"test rel L2 @256             = {res['test_rel_l2_256']:.3%}")
for name, r in res["refine"].items():
    print(f"refine {name:16s} {r['rel_l2_before']:.1%} -> {r['rel_l2_after']:.1%}")

In [ ]:
from IPython.display import Image

# on the boosted (OOD) preset, physics refinement walks the shock to the right place
Image(filename="checkpoints/pino/figures/refine_presets.png")

**Takeaway.** The PINO cuts the physics residual ~5× below the FNO, and refinement
rescues badly out-of-distribution inputs with no data at all — closing the loop
from the PINN (physics, slow) to the FNO (data, fast) and back.